# Week 10 – Streaming Simulation

## Objective
Simulate JSON event streaming using Auto Loader and Structured Streaming.

## Activities
- Create JSON event files
- Read arriving JSON files incrementally using Auto Loader
- Simulate live events
- Process events using Structured Streaming
- Display one live metric
- Understand Kafka-style event design without installing Kafka

## Evidence
The notebook will capture incremental input, streaming output, and a live metric.

In [0]:
spark.sql("SHOW CATALOGS").show()

+---------+
|  catalog|
+---------+
|  cityfix|
|  samples|
|   system|
|workspace|
+---------+



In [0]:
spark.sql("SHOW SCHEMAS IN cityfix").show()

+------------------+
|      databaseName|
+------------------+
|           cityfix|
|           default|
|              gold|
|information_schema|
+------------------+



In [0]:
spark.sql("USE CATALOG cityfix")

DataFrame[]

In [0]:
spark.sql("SHOW SCHEMAS").show()

+------------------+
|      databaseName|
+------------------+
|           cityfix|
|           default|
|              gold|
|information_schema|
+------------------+



In [0]:
spark.sql("SHOW TABLES IN cityfix.default").show(truncate=False)

+--------+---------+-----------+
|database|tableName|isTemporary|
+--------+---------+-----------+
+--------+---------+-----------+



In [0]:
spark.sql("SHOW TABLES IN cityfix.default").show(truncate=False)

+--------+---------+-----------+
|database|tableName|isTemporary|
+--------+---------+-----------+
+--------+---------+-----------+



In [0]:
spark.sql("SHOW TABLES IN cityfix.gold").show(truncate=False)

+--------+--------------------------------+-----------+
|database|tableName                       |isTemporary|
+--------+--------------------------------+-----------+
|gold    |agency_sla_summary              |false      |
|gold    |borough_backlog_summary         |false      |
|gold    |category_volume_summary         |false      |
|gold    |channel_and_location_summary    |false      |
|gold    |dim_agency                      |false      |
|gold    |dim_channel                     |false      |
|gold    |dim_complaint_category          |false      |
|gold    |dim_date                        |false      |
|gold    |dim_geography_borough_zip       |false      |
|gold    |dim_location_type               |false      |
|gold    |dim_request_status              |false      |
|gold    |dim_time_band                   |false      |
|gold    |fact_request_backlog_snapshot   |false      |
|gold    |fact_request_event_stream       |false      |
|gold    |fact_request_resolution         |false

In [0]:
spark.sql("DESCRIBE cityfix.gold.fact_request_event_stream").show(truncate=False)

+----------------------+---------+-------+
|col_name              |data_type|comment|
+----------------------+---------+-------+
|event_id              |string   |NULL   |
|event_ts              |timestamp|NULL   |
|unique_key            |string   |NULL   |
|event_type            |string   |NULL   |
|agency_key            |bigint   |NULL   |
|complaint_category_key|bigint   |NULL   |
|geography_key         |bigint   |NULL   |
|request_status_key    |bigint   |NULL   |
|channel_key           |bigint   |NULL   |
|location_type_key     |bigint   |NULL   |
|payload               |string   |NULL   |
+----------------------+---------+-------+



In [0]:
spark.sql("""
SELECT event_id, event_ts, unique_key, event_type,
       agency_key, complaint_category_key, geography_key,
       request_status_key, channel_key, location_type_key, payload
FROM cityfix.gold.fact_request_event_stream
LIMIT 5
""").show(truncate=False)

+--------+--------+----------+----------+----------+----------------------+-------------+------------------+-----------+-----------------+-------+
|event_id|event_ts|unique_key|event_type|agency_key|complaint_category_key|geography_key|request_status_key|channel_key|location_type_key|payload|
+--------+--------+----------+----------+----------+----------------------+-------------+------------------+-----------+-----------------+-------+
+--------+--------+----------+----------+----------+----------------------+-------------+------------------+-----------+-----------------+-------+



In [0]:
spark.sql("SHOW TABLES IN cityfix").show(truncate=False)

+--------+-------------------------+-----------+
|database|tableName                |isTemporary|
+--------+-------------------------+-----------+
|cityfix |bronze_agencies          |false      |
|cityfix |bronze_boroughs          |false      |
|cityfix |bronze_categories        |false      |
|cityfix |bronze_requests          |false      |
|cityfix |bronze_zip_geography     |false      |
|cityfix |quarantine_requests      |false      |
|cityfix |replay_closure_evidence  |false      |
|cityfix |silver_candidate_requests|false      |
|cityfix |silver_trusted_requests  |false      |
+--------+-------------------------+-----------+



In [0]:
spark.sql("""
DESCRIBE cityfix.bronze_requests
""").show(truncate=False)

+----------------------+---------+-------+
|col_name              |data_type|comment|
+----------------------+---------+-------+
|physical_record_key   |string   |NULL   |
|unique_key            |string   |NULL   |
|created_date          |string   |NULL   |
|due_date              |string   |NULL   |
|closed_date           |string   |NULL   |
|agency_code           |string   |NULL   |
|agency_name_raw       |string   |NULL   |
|complaint_category    |string   |NULL   |
|complaint_type        |string   |NULL   |
|descriptor            |string   |NULL   |
|borough_code          |string   |NULL   |
|borough               |string   |NULL   |
|zip_code              |string   |NULL   |
|latitude              |string   |NULL   |
|longitude             |string   |NULL   |
|location_type         |string   |NULL   |
|channel               |string   |NULL   |
|status                |string   |NULL   |
|priority              |string   |NULL   |
|resolution_description|string   |NULL   |
+----------

In [0]:
spark.sql("""
SELECT unique_key, created_date, agency_code,
       complaint_category, complaint_type,
       borough, channel, status, priority, location_type
FROM cityfix.bronze_requests
LIMIT 5
""").show(truncate=False)

+------------------+--------------------+-----------+------------------------+---------------------+---------------+-----------+------+--------+------------------+
|unique_key        |created_date        |agency_code|complaint_category      |complaint_type       |borough        |channel    |status|priority|location_type     |
+------------------+--------------------+-----------+------------------------+---------------------+---------------+-----------+------+--------+------------------+
|CFX-2025-000000001|2025-05-07T17:19:07Z|HLT        |Public Health           |Mosquito Breeding    |West Borough   |Web Portal |Closed|Normal  |Street            |
|CFX-2025-000000002|2025-11-14T15:59:38Z|ENV        |Environment and Nuisance|Noise Nuisance       |West Borough   |Mobile App |Closed|High    |Municipal Building|
|CFX-2025-000000003|2025-08-11T17:54:26Z|PWD        |Roads and Footpaths     |Streetlight Fault    |South Borough  |Web Portal |Closed|Normal  |Municipal Building|
|CFX-2025-000000

In [0]:
import json

event = {
    "event_id": "W10-CFX-001",
    "event_ts": "2026-10-06T10:00:00Z",
    "unique_key": "CFX-2025-000000001",
    "event_type": "request_update",
    "agency_key": 1,
    "complaint_category_key": 1,
    "geography_key": 1,
    "request_status_key": 1,
    "channel_key": 1,
    "location_type_key": 1,
    "payload": json.dumps({
        "agency_code": "HLT",
        "complaint_category": "Public Health",
        "complaint_type": "Mosquito Breeding",
        "borough": "West Borough",
        "channel": "Web Portal",
        "status": "Closed",
        "priority": "Normal",
        "location_type": "Street"
    })
}

print(json.dumps(event, indent=2))

{
  "event_id": "W10-CFX-001",
  "event_ts": "2026-10-06T10:00:00Z",
  "unique_key": "CFX-2025-000000001",
  "event_type": "request_update",
  "agency_key": 1,
  "complaint_category_key": 1,
  "geography_key": 1,
  "request_status_key": 1,
  "channel_key": 1,
  "location_type_key": 1,
  "payload": "{\"agency_code\": \"HLT\", \"complaint_category\": \"Public Health\", \"complaint_type\": \"Mosquito Breeding\", \"borough\": \"West Borough\", \"channel\": \"Web Portal\", \"status\": \"Closed\", \"priority\": \"Normal\", \"location_type\": \"Street\"}"
}


In [0]:
spark.sql("SHOW VOLUMES IN cityfix.cityfix").show(truncate=False)

+--------+--------------+
|database|volume_name   |
+--------+--------------+
|cityfix |cityfix-zenaiz|
+--------+--------------+



In [0]:
display(dbutils.fs.ls("/Volumes/cityfix/cityfix/cityfix-zenaiz"))

path,name,size,modificationTime
dbfs:/Volumes/cityfix/cityfix/cityfix-zenaiz/complaints_sample.csv,complaints_sample.csv,127,1784956663000
dbfs:/Volumes/cityfix/cityfix/cityfix-zenaiz/departments_sample.csv,departments_sample.csv,71,1784956678000
dbfs:/Volumes/cityfix/cityfix/cityfix-zenaiz/raw/,raw/,0,1791291875953
dbfs:/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/,streaming/,0,1791291875953
dbfs:/Volumes/cityfix/cityfix/cityfix-zenaiz/wards_sample.csv,wards_sample.csv,55,1784956678000


In [0]:
display(dbutils.fs.ls("/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming"))

path,name,size,modificationTime
dbfs:/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/new_request_event.json,new_request_event.json,4705,1785500782000
dbfs:/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/new_request_event_drop_02.json,new_request_event_drop_02.json,4766,1785500782000
dbfs:/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/new_request_event_drop_03.json,new_request_event_drop_03.json,4252,1785500782000
dbfs:/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/new_request_event_drop_04.json,new_request_event_drop_04.json,2955,1785500782000
dbfs:/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/new_request_event_drop_05.json,new_request_event_drop_05.json,4775,1785500782000
dbfs:/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/new_request_event_drop_06.json,new_request_event_drop_06.json,3795,1785500782000


In [0]:
display(dbutils.fs.ls("/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming"))

path,name,size,modificationTime
dbfs:/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/new_request_event.json,new_request_event.json,4705,1785500782000
dbfs:/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/new_request_event_drop_02.json,new_request_event_drop_02.json,4766,1785500782000
dbfs:/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/new_request_event_drop_03.json,new_request_event_drop_03.json,4252,1785500782000
dbfs:/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/new_request_event_drop_04.json,new_request_event_drop_04.json,2955,1785500782000
dbfs:/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/new_request_event_drop_05.json,new_request_event_drop_05.json,4775,1785500782000
dbfs:/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/new_request_event_drop_06.json,new_request_event_drop_06.json,3795,1785500782000


In [0]:
display(dbutils.fs.head(
    "/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/new_request_event.json",
    5000
))

'{"request_event_id":"EVT-01-0001","unique_key":"CFX-2026-000181000","event_type":"NEW_REQUEST","event_timestamp":"2026-07-17T17:00:00Z","status":"Open","agency_code":"PWD","complaint_category":"Roads and Footpaths","complaint_type":"Pothole","descriptor":"Road surface damage","borough":"North Borough","zip_code":"510001","latitude":17.4935,"longitude":78.4385,"channel":"Web Portal","priority":"Normal","sla_due_timestamp":"2026-07-20T17:00:00Z","source_system":"CITYFIX_STREAM_SIM","schema_version":"1.0","event_sequence":1,"ingestion_timestamp":"2026-07-18T00:30:00Z"}\n{"request_event_id":"EVT-01-0002","unique_key":"CFX-2026-000181001","event_type":"NEW_REQUEST","event_timestamp":"2026-07-17T17:10:00Z","status":"Open","agency_code":"PWD","complaint_category":"Roads and Footpaths","complaint_type":"Footpath Damage","descriptor":"Broken or obstructed footpath","borough":"North Borough","zip_code":"510006","latitude":17.5345,"longitude":78.4715,"channel":"Web Portal","priority":"Normal","s

In [0]:
display(
    spark.read.json(
        "/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/*.json"
    )
)

_corrupt_record,agency_code,borough,channel,citizen_language,complaint_category,complaint_type,descriptor,event_sequence,event_timestamp,event_type,ingestion_timestamp,latitude,longitude,priority,request_event_id,schema_version,sensor_confidence,sla_due_timestamp,source_system,status,unique_key,ward_code,zip_code
null,PKS,Central Borough,Mobile App,null,Parks and Public Space,Fallen Tree or Branch,Tree or branch obstructing area,2,2026-07-17T23:00:00Z,STATUS_UPDATE,2026-07-18T00:30:00Z,17.464456,78.533551,High,EVT-05-0001,1.0,null,2026-07-19T19:00:00Z,CITYFIX_STREAM_SIM,In Progress,CFX-2025-999999999,null,510048
null,ZZZ,North Borough,Web Portal,null,Roads and Footpaths,Footpath Damage,Broken or obstructed footpath,1,2026-07-17T23:05:00Z,NEW_REQUEST,2026-07-18T00:30:00Z,17.5345,78.4715,Normal,EVT-05-0002,1.0,null,2026-07-22T17:10:00Z,CITYFIX_STREAM_SIM,Open,CFX-2026-000181030,null,510006
null,PWD,East Borough,Web Portal,null,Unknown Civic Type,Streetlight Fault,Lamp not working or flickering,1,2026-07-17T23:10:00Z,NEW_REQUEST,2026-07-18T00:30:00Z,17.4155,78.6185,Normal,EVT-05-0003,1.0,null,2026-07-19T17:20:00Z,CITYFIX_STREAM_SIM,Open,CFX-2026-000181031,null,510023
null,SAN,East Borough,Web Portal,null,Waste and Cleanliness,Missed Waste Collection,Scheduled collection missed,1,2026-07-17T23:15:00Z,NEW_REQUEST,2026-07-18T00:30:00Z,17.4565,78.6515,Normal,EVT-05-0004,1.0,null,2026-07-18T17:30:00Z,CITYFIX_STREAM_SIM,Archived Secretly,CFX-2026-000181032,null,510028
null,PWD,Central Borough,Web Portal,null,Roads and Footpaths,Streetlight Fault,Lamp not working or flickering,2,2025-12-31T22:58:04Z,STATUS_UPDATE,2026-07-18T00:30:00Z,17.447714,78.465589,High,EVT-05-0005,1.0,null,2026-07-18T03:00:00Z,CITYFIX_STREAM_SIM,Closed,CFX-2025-000009285,null,510045
null,SAN,Central Borough,Web Portal,null,Waste and Cleanliness,Illegal Dumping,Waste dumped in public area,1,2026-07-17T23:25:00Z,NEW_REQUEST,2026-07-18T00:30:00Z,95.0,78.4565,Normal,EVT-05-0006,1.0,null,2026-07-19T17:40:00Z,CITYFIX_STREAM_SIM,Open,CFX-2026-000181033,null,510045
null,SAN,South Borough,Web Portal,null,Waste and Cleanliness,Overflowing Bin,Public bin requires service,1,2026-07-19T00:00:00Z,NEW_REQUEST,2026-07-18T00:30:00Z,17.3095,78.4535,Normal,EVT-05-0007,1.0,null,2026-07-18T05:50:00Z,CITYFIX_STREAM_SIM,Open,CFX-2026-000181034,null,510062
null,WAT,South Borough,Web Portal,null,Water and Sewer,Water Leakage,Visible supply-line leakage,1,2026-07-17T23:35:00Z,NEW_REQUEST,2026-07-18T00:30:00Z,17.3505,78.4865,Normal,EVT-05-0008,1.0,null,2026-07-18T06:00:00Z,CITYFIX_STREAM_SIM,Open,CFX-2026-000181035,null,510067
null,WAT,Central Borough,Call Centre,null,Water and Sewer,Water Leakage,Visible supply-line leakage,2,2026-07-17T18:00:00Z,STATUS_UPDATE,2026-07-18T00:30:00Z,17.450843,78.465625,High,EVT-02-0001,1.0,null,2026-07-18T23:00:00Z,CITYFIX_STREAM_SIM,In Progress,CFX-2025-000130997,null,510045
null,HLT,North Borough,Web Portal,null,Public Health,Public Toilet Maintenance,Cleaning or repair needed,2,2026-07-17T18:08:00Z,STATUS_UPDATE,2026-07-18T00:30:00Z,17.536382,78.438554,Normal,EVT-02-0002,1.0,null,2026-07-18T23:00:00Z,CITYFIX_STREAM_SIM,In Progress,CFX-2025-000154403,null,510005


In [0]:
for f in dbutils.fs.ls("/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming"):
    print(f.name)

new_request_event.json
new_request_event_drop_02.json
new_request_event_drop_03.json
new_request_event_drop_04.json
new_request_event_drop_05.json
new_request_event_drop_06.json


In [0]:
from pyspark.sql.functions import *

stream_path = "/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/"

stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", "/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/_schema")
    .load(stream_path)
)

display(stream_df)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6770307904028602>, line 14
      3 stream_path = "/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/"
      5 stream_df = (
      6     spark.readStream
      7     .format("cloudFiles")
   (...)     11     .load(stream_path)
     12 )
---> 14 display(stream_df)

File <command-6770307904028602>, line 6
      1 from pyspark.sql.functions import *
      3 stream_path = "/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/"
      5 stream_df = (
----> 6     spark.readStream
      7     .format("cloudFiles")
      8     .option("cloudFiles.format", "json")
      9     .option("cloudFiles.inferColumnTypes", "true")
     10     .option("cloudFiles.schemaLocation", "/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/_schema")
     11     .load(stream_path)
     12 )
     14 display(stream_df)

AnalysisException: [TEMP_CHECKPOINT_LOC

In [0]:
from pyspark.sql.functions import *

stream_path = "/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/"
checkpoint_path = "/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/_checkpoint"

stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", "/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/_schema")
    .load(stream_path)
)

display(
    stream_df,
    checkpointLocation=checkpoint_path
)

Checkpointing to /Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/_checkpoint


In [0]:
from pyspark.sql.functions import count

live_count = stream_df.agg(
    count("*").alias("live_event_count")
)

display(
    live_count,
    checkpointLocation="/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/_count_checkpoint"
)

Checkpointing to /Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/_count_checkpoint


In [0]:
spark.streams.active

[]

In [0]:
dbutils.fs.mkdirs(
    "/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/week10_input"
)

True

In [0]:
dbutils.fs.cp(
    "/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/new_request_event.json",
    "/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/week10_input/drop_01.json"
)

True

In [0]:
clean_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.inferColumnTypes", "true")
    .option(
        "cloudFiles.schemaLocation",
        "/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/week10_schema"
    )
    .load(
        "/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/week10_input"
    )
)

display(
    clean_stream,
    checkpointLocation="/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/week10_checkpoint"
)

Checkpointing to /Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/week10_checkpoint


In [0]:
display(
    clean_stream,
    checkpointLocation="/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/week10_checkpoint"
)

Checkpointing to /Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/week10_checkpoint


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6890749913504057>, line 1
----> 1 display(
      2     clean_stream,
      3     checkpointLocation="/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/week10_checkpoint"
      4 )

File <command-6890749913504056>, line 2
      1 clean_stream = (
----> 2     spark.readStream
      3     .format("cloudFiles")
      4     .option("cloudFiles.format", "json")
      5     .option("cloudFiles.inferColumnTypes", "true")
      6     .option(
      7         "cloudFiles.schemaLocation",
      8         "/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/week10_schema"
      9     )
     10     .load(
     11         "/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/week10_input"
     12     )
     13 )
     15 display(
     16     clean_stream,
     17     checkpointLocation="/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/week10

In [0]:
from pyspark.sql.functions import count

clean_live_count = clean_stream.agg(
    count("*").alias("live_event_count")
)

display(
    clean_live_count,
    checkpointLocation="/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/week10_count_checkpoint"
)

Checkpointing to /Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/week10_count_checkpoint


In [0]:
dbutils.fs.cp(
    "/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/new_request_event_drop_02.json",
    "/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/week10_input/drop_02.json"
)

True

In [0]:
display(
    clean_live_count,
    checkpointLocation="/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/week10_count_checkpoint"
)

Checkpointing to /Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/week10_count_checkpoint


In [0]:
[(q.id, q.status) for q in spark.streams.active]

[]

In [0]:
clean_live_count = clean_stream.agg(
    count("*").alias("live_event_count")
)

display(
    clean_live_count,
    checkpointLocation="/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/week10_count_checkpoint_02"
)

Checkpointing to /Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/week10_count_checkpoint_02


In [0]:
[(q.id, q.isActive, q.status) for q in spark.streams.active]

[]

In [0]:
query = (
    clean_stream.writeStream
    .format("memory")
    .queryName("week10_live_events")
    .outputMode("append")
    .trigger(availableNow=True)
    .option(
        "checkpointLocation",
        "/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/week10_live_checkpoint"
    )
    .start()
)

query.awaitTermination()

print("Streaming batch completed")

Streaming batch completed


In [0]:
spark.sql("""
SELECT COUNT(*) AS processed_event_count
FROM week10_live_events
""").show()

+---------------------+
|processed_event_count|
+---------------------+
|                   16|
+---------------------+



In [0]:
spark.sql("""
SELECT *
FROM week10_live_events
LIMIT 10
""").show(truncate=False)

+-----------+---------------+-----------+---------------------+-------------------------+-----------------------------+--------------+--------------------+-------------+--------------------+---------+---------+--------+----------------+--------------+--------------------+------------------+-----------+------------------+--------+-------------+
|agency_code|borough        |channel    |complaint_category   |complaint_type           |descriptor                   |event_sequence|event_timestamp     |event_type   |ingestion_timestamp |latitude |longitude|priority|request_event_id|schema_version|sla_due_timestamp   |source_system     |status     |unique_key        |zip_code|_rescued_data|
+-----------+---------------+-----------+---------------------+-------------------------+-----------------------------+--------------+--------------------+-------------+--------------------+---------+---------+--------+----------------+--------------+--------------------+------------------+-----------+-----

In [0]:
spark.sql("""
SELECT
    event_type,
    COUNT(*) AS event_count
FROM week10_live_events
GROUP BY event_type
ORDER BY event_type
""").show(truncate=False)

+-------------+-----------+
|event_type   |event_count|
+-------------+-----------+
|NEW_REQUEST  |10         |
|STATUS_UPDATE|6          |
+-------------+-----------+



In [0]:
dbutils.fs.cp(
    "/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/new_request_event_drop_03.json",
    "/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/week10_input/drop_03.json"
)

True

In [0]:
incremental_query = (
    clean_stream.writeStream
    .format("memory")
    .queryName("week10_incremental_events")
    .outputMode("append")
    .trigger(availableNow=True)
    .option(
        "checkpointLocation",
        "/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/week10_incremental_checkpoint"
    )
    .start()
)

incremental_query.awaitTermination()

print("Incremental streaming completed")

Incremental streaming completed


In [0]:
spark.sql("""
SELECT COUNT(*) AS incremental_event_count
FROM week10_incremental_events
""").show()

+-----------------------+
|incremental_event_count|
+-----------------------+
|                     23|
+-----------------------+



In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/week10_input"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/week10_input/drop_01.json,drop_01.json,4705,1791296475000
dbfs:/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/week10_input/drop_02.json,drop_02.json,4766,1791296696000
dbfs:/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/week10_input/drop_03.json,drop_03.json,4252,1791297039000


In [0]:
id="q2v8pk"
spark.read.json(
    "/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/week10_input/drop_03.json"
).select(
    "request_event_id",
    "unique_key",
    "event_type",
    "event_timestamp",
    "status"
).show(truncate=False)

+----------------+------------------+-------------+--------------------+-----------+
|request_event_id|unique_key        |event_type   |event_timestamp     |status     |
+----------------+------------------+-------------+--------------------+-----------+
|EVT-03-0001     |CFX-2025-000044094|STATUS_UPDATE|2026-07-17T20:00:00Z|In Progress|
|EVT-03-0002     |CFX-2025-000073638|STATUS_UPDATE|2026-07-17T20:10:00Z|In Progress|
|EVT-03-0003     |CFX-2025-000174256|STATUS_UPDATE|2026-07-17T20:20:00Z|In Progress|
|EVT-03-0004     |CFX-2025-000044094|STATUS_UPDATE|2026-07-17T20:40:00Z|Open       |
|EVT-03-0005     |CFX-2025-000015724|STATUS_UPDATE|2026-07-17T13:00:00Z|In Progress|
|EVT-03-0006     |CFX-2026-000181010|NEW_REQUEST  |2026-07-17T20:50:00Z|Open       |
|EVT-03-0007     |CFX-2026-000181011|NEW_REQUEST  |2026-07-17T20:55:00Z|Open       |
+----------------+------------------+-------------+--------------------+-----------+



In [0]:
spark.read.json(
    "/Volumes/cityfix/cityfix/cityfix-zenaiz/streaming/week10_input/drop_03.json"
).groupBy("event_type").count().orderBy("event_type").show()

+-------------+-----+
|   event_type|count|
+-------------+-----+
|  NEW_REQUEST|    2|
|STATUS_UPDATE|    5|
+-------------+-----+



In [0]:
spark.sql("""
SELECT
    event_type,
    COUNT(*) AS event_count
FROM week10_incremental_events
GROUP BY event_type
ORDER BY event_type
""").show()

+-------------+-----------+
|   event_type|event_count|
+-------------+-----------+
|  NEW_REQUEST|         12|
|STATUS_UPDATE|         11|
+-------------+-----------+



In [0]:
spark.sql("""
SELECT
    COUNT(*) AS total_events,
    COUNT(DISTINCT unique_key) AS unique_requests
FROM week10_incremental_events
""").show()

+------------+---------------+
|total_events|unique_requests|
+------------+---------------+
|          23|             20|
+------------+---------------+



In [0]:
display(
    spark.sql("""
        SELECT
            event_type,
            COUNT(*) AS event_count
        FROM week10_incremental_events
        GROUP BY event_type
        ORDER BY event_type
    """)
)

event_type,event_count
NEW_REQUEST,12
STATUS_UPDATE,11


In [0]:
spark.sql("""
SELECT
    request_event_id,
    unique_key,
    event_type,
    event_timestamp,
    status
FROM week10_incremental_events
ORDER BY event_timestamp
LIMIT 10
""").show(truncate=False)

+----------------+------------------+-------------+--------------------+-----------+
|request_event_id|unique_key        |event_type   |event_timestamp     |status     |
+----------------+------------------+-------------+--------------------+-----------+
|EVT-03-0005     |CFX-2025-000015724|STATUS_UPDATE|2026-07-17T13:00:00Z|In Progress|
|EVT-01-0001     |CFX-2026-000181000|NEW_REQUEST  |2026-07-17T17:00:00Z|Open       |
|EVT-01-0002     |CFX-2026-000181001|NEW_REQUEST  |2026-07-17T17:10:00Z|Open       |
|EVT-01-0003     |CFX-2026-000181002|NEW_REQUEST  |2026-07-17T17:20:00Z|Open       |
|EVT-01-0004     |CFX-2026-000181003|NEW_REQUEST  |2026-07-17T17:30:00Z|Open       |
|EVT-01-0005     |CFX-2026-000181004|NEW_REQUEST  |2026-07-17T17:40:00Z|Open       |
|EVT-01-0006     |CFX-2026-000181005|NEW_REQUEST  |2026-07-17T17:50:00Z|Open       |
|EVT-02-0001     |CFX-2025-000130997|STATUS_UPDATE|2026-07-17T18:00:00Z|In Progress|
|EVT-02-0001     |CFX-2025-000130997|STATUS_UPDATE|2026-07-17T18: